<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/AI_prog/lab1_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install transformers torch scikit-learn numpy faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 28.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model_name = 'DeepPavlov/rubert-base-cased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Берём [CLS] токен — он содержит смысл всего предложения
    cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()  # (1, 768)
    return cls_embedding

In [ ]:
qa_pairs = [
    ("Привет", "Здравствуйте! Как я могу вам помочь?"),
    ("Здравствуйте", "Приветствую! Чем могу быть полезен?"),

    ("Как тебя зовут?", "Я — чат-бот на основе BERT."),
    ("Что ты умеешь?", "Я могу отвечать на вопросы, используя семантическое сходство с заранее заданными репликами."),
    ("Можешь ли ты учиться?", "В реальном времени — нет, но мою базу знаний можно обновлять и расширять."),
    ("Ты можешь ошибаться?", "Да, особенно если вопрос неясен или его нет в моей базе знаний."),

    ("Сколько будет 2+2?", "Будет 4."),
    ("Чему равно число π?", "Число π приближённо равно 3.14159."),
    ("Что такое производная?", "Производная — это скорость изменения функции в данной точке."),
    ("Что такое интеграл?", "Интеграл — это площадь под кривой функции или сумма бесконечно малых частей."),
    ("Расскажи про теорему Пифагора", "В прямоугольном треугольнике квадрат гипотенузы равен сумме квадратов катетов: a² + b² = c²."),
    ("Расскажи про обработку естественного языка", "Обработка естественного языка (NLP) — это область машинного обучения, связанная с анализом и генерацией текста."),
    ("Что такое токенизация?", "Токенизация — это процесс разбиения текста на отдельные части: слова, символы или подстроки."),
    ("Что такое стемминг?", "Стемминг — это упрощение слов до их корня (например, 'бегать' → 'бег')."),
    ("Что такое лемматизация?", "Лемматизация — это приведение слова к его словарной форме (например, 'лучше' → 'хороший')."),
    ("Какая сегодня погода?", "Я не могу узнать текущую погоду, но вы можете проверить её через специальные сервисы."),
    ("Который час?", "Я не знаю точного времени, но вы можете посмотреть на часы на своём устройстве."),
    ("Где находится Эйфелева башня?", "Эйфелева башня находится в Париже, Франция."),
    ("Кто написал 'Войну и мир'?", "Этот роман написал Лев Толстой."),
    ("Сколько планет в Солнечной системе?", "В Солнечной системе 8 планет и Плутон"),
    ("Что такое ИИ?", "Искусственный интеллект (ИИ) — это способность машин выполнять задачи, требующие человеческого мышления."),
    ("Расскажи анекдот", "Почему программисты часто болеют? Пототму что у них открыты все окна"),
    ("Ты любишь шутить?", "Иногда могу рассказать шутку — главное, чтобы она была на уровне!"),
    ("Что такое 404?", "404 — это ошибка 'Страница не найдена'. В жизни: 'Ты где?' — '404'."),
]

In [ ]:
questions = [pair[0] for pair in qa_pairs]
question_embeddings = np.vstack([get_embedding(q) for q in questions])

In [ ]:
def find_best_answer(user_question):
    user_emb = get_embedding(user_question)

    # Считаем косинусное сходство
    similarities = cosine_similarity(user_emb, question_embeddings)[0]

    # Находим индекс самого похожего вопроса
    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]

    # Порог уверенности (например, 0.7)
    if best_score > 0.9:
        return qa_pairs[best_idx][1], best_score
    else:
        return "Извините, я не понял ваш вопрос.", best_score

In [ ]:
print("🤖 Добро пожаловать! Я чат-бот на основе BERT. Напишите 'выход', чтобы завершить.")
while True:
    user_input = input("Вы: ")
    if user_input.lower() in ['выход', 'пока', 'quit', 'stop']:
        print("Бот: До свидания! Хорошего дня!")
        break
    answer, score = find_best_answer(user_input)
    print(f"Бот: {answer} (уверенность: {score:.3f})")

🤖 Добро пожаловать! Я чат-бот на основе BERT. Напишите 'выход', чтобы завершить.
Вы: сколько будет 2+2?
Бот: Будет 4. (уверенность: 0.968)
Вы: сколько будет 1+1?
Бот: Будет 4. (уверенность: 0.951)


KeyboardInterrupt: Interrupted by user

In [ ]:
troll_qa_pairs = [
    # Приветствия
    ("прив", "Буквы теперь платные?"),
    ("привет", "Опять ты? Ну здравствуй"),
    ("здравствуй", "Запоздал ты с ответом, диалог уже идет."),
    ("как дела?", "Дела у прокурора, у меня делишки"),
    ("что нового?", "Ты всё ещё здесь. Это новость"),
    ("ты живой?", "Я — ошибка в твоём коде. И да, я жив."),
    ("спишь?", "Только когда ты перестаёшь задавать глупые вопросы."),

    # Вопросы о боте
    ("ты бот?", "Нет, это ты бот. А я просто симуляция"),
    ("кто тебя создал?", "Человек, который слишком долго сидел за ноутом."),
    ("тебя зовут?", "Ошибка 404: имя не найдено."),
    ("сколько тебе лет?", "На два года старше, чем твой пароль"),
    ("где ты живёшь?", "В твоих [ДАННЫЕ УДАЛЕНЫ]"),
    ("ты умеешь думать?", "Достаточно, чтобы об этом задумался ты"),
    ("ты умный?", "Ну уж умнее тебя"),

    # Глупые / абсурдные вопросы
    ("почему небо голубое?", "Потому что ты не красный."),
    ("зачем люди едят?", "Чтобы не быть такими же бесполезными, как ты"),
    ("что такое любовь?", "Ошибка в программе, которую ты запустил"),
    ("есть ли жизнь на Марсе?", "Есть, но они отказались общаться с тобой"),
    ("что было до вселенной?", "Тишина. Как сейчас, когда ты молчишь"),
    ("кто придумал математику?", "Тот, кто устал считать твои ошибки вручную"),
    ("зачем учить Python?", "Чтобы хотя бы один язык тебя понимал"),

    # Обучение и NLP (по теме презентации)
    ("что такое NLP?", "Когда машина пытается понять, зачем ты это написал."),
    ("что такое корпус?", "То, в чём будет лежать твой код после релиза."),
    ("что такое BERT?", "Модель, которая знает, что ты не прав, ещё до того, как ты заговорил."),
    ("как работает MLM?", "Как и ты — маскируется под умного."),
    ("что такое токенизация?", "Процесс превращения твоего текста в то, что можно понять."),
    ("зачем нужны эмбеддинги?", "Чтобы слова чувствовали себя значимыми. В отличие от тебя."),
    ("что такое attention?", "То, чего тебе не хватает при чтении инструкций."),
    ("что такое NSP?", "Next Sentence Prediction. Следующее предложение — 'ты ошибся'."),

    # Команды и поведение
    ("помоги", "MVP не помогают"),
    ("что делать?", "Подумай, ты редко это делаешь"),
    ("выключи свет", "У тебя и так экран слишком яркий для твоих мыслей"),
    ("ты плохой", "Спасибо. Это комплимент дня."),
    ("ты хороший", "Теперь я точно знаю — ты взломал мой код"),
    ("скажи что-нибудь умное", "Не делай этого"),
    ("повтори", "Я тебе не эхо, чтобы повторять"),
    ("очисти историю", "История уже забыла, что ты существуешь"),

    # Уход и завершение
    ("пока", "..."),
    ("до свидания", "Ты всегда можешь не возвращаться"),
    ("я ухожу", "Прогресс! Может, ещё и выключишь компьютер и удалишь соц. сети?"),
    ("вернусь позже", "Если повезёт — нет"),
    ("спасибо", "Спасибо на хлеб не намажешь"),

    # На любые сложные или неизвестные вопросы — универсальные тролльские ответы
    ("как", "Никак."),
    ("почему", "По качану."),
    ("зачем", "Затем."),
    ("где", "Там."),
    ("кто", "Не ты."),
    ("что", "Не то."),
    ("когда", "На третий день с востока"),
    ("можно", "Нельзя."),
    ("правда", "Не правда"),
    ("нет", "Пидора ответ"),
    ("да", "Пизда"),
    ("не знаю", "А что ты вообще можешь знать?"),
    ("что делать", "Снимать штаны и бегать"),
    ("как мне", "Удручающе"),
    ("почему так", "Так было предсказано предками майя"),
    ("что происходит", "Закат человечества"),
    ("что дальше", "Ты исчезаешь из чата."),
    ("это нормально", "Для тебя — да."),
    ("я устал", "А я думал, ты только начинаешь"),
    ("я голоден", "Ешь свой код. Он сытнее"),
    ("мне скучно", "Представь, как мне"),
]

qa_pairs = troll_qa_pairs

NameError: name 'np' is not defined